# Data Preparation and Feature Engineering

This notebook implements the **Data Preparation** phase of the CRISP-DM methodology for the *Mental Health in Technology* project.

Notebook 01 established the characteristics and principal data-quality issues of the OSMI Mental Health in Tech 2016 survey. It found 1,433 observations and 63 variables, with mixed numerical and categorical data, substantial and non-uniform missingness, high-cardinality responses, and free-text fields.

This notebook therefore does **not repeat the exploratory data audit** performed previously. Instead, it uses those findings as the evidence base for preparing the survey for subsequent feature selection, dimensionality reduction, and clustering.

The preparation process focuses on:

- preserving the original raw dataset;
- applying justified data-quality corrections;
- distinguishing structured survey responses from free text;
- engineering meaningful indicators from multi-response fields;
- preserving genuine binary and categorical information;
- handling missing values according to variable type and survey structure;
- creating a reproducible analytical feature matrix.

The resulting representation will be used by the subsequent analytical stages.

No clustering model is fitted in this notebook.

## 1. Preparation Objectives

### Question

How should the raw survey responses be transformed into a defensible analytical representation for unsupervised machine learning?

### Method

The preparation decisions follow directly from the evidence established during Data Understanding.

The main decisions are:

1. Correct values that are demonstrably invalid representations of their variables.
2. Preserve the original raw observations and create prepared representations separately.
3. Exclude raw free-text fields from the primary structured feature representation.
4. Avoid treating high-cardinality condition text as ordinary categorical variables.
5. Decompose the multi-response work-position field into interpretable binary indicators.
6. Preserve binary variables as binary information rather than imposing arbitrary numerical rankings.
7. Treat categorical missingness explicitly rather than replacing it indiscriminately with the most frequent category.
8. Impute the continuous age variable only after invalid values have been converted to missing.
9. Avoid arbitrary ordinal encoding where a complete and defensible ordering has not been established.
10. Keep the resulting representation suitable for later feature selection and dimensionality reduction.

### Why

Clustering operates on the numerical representation supplied to the algorithm. Consequently, data-preparation decisions directly affect distances, similarity relationships, and the resulting cluster structure.

The objective is therefore not to maximise the number of features, but to construct a representation that preserves meaningful survey information while avoiding artificial structure and unnecessary dimensionality.

In [1]:
# Import the libraries required for data loading, numerical validation, and preprocessing.
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from mental_health_ml.cleaning import clean_age
from mental_health_ml.data import load_dataset
from mental_health_ml.feature_engineering import engineer_work_position_features

pd.set_option("display.max_columns", None)

## 2. Load the Raw Survey Data

Notebook 01 established the raw dataset location and structure.

The original CSV is loaded again here because each notebook should be independently reproducible while maintaining the same raw-data boundary.

The raw dataset is treated as immutable. All preparation operations create a separate analytical representation.

In [2]:
# Load the immutable raw survey dataset established during Data Understanding.
df = load_dataset(
    Path("../data/raw/mental-heath-in-tech-2016_20161114.csv.zip")
)

print(
    f"Raw dataset loaded: {df.shape[0]:,} observations "
    f"x {df.shape[1]:,} variables."
)

Raw dataset loaded: 1,433 observations x 63 variables.


## 3. Raw-Data Integrity Check

The raw-data dimensions established in Notebook 01 are used as a reproducibility check.

This is not a second exploratory analysis. It is a guard against accidentally preparing the wrong file or an already modified dataset.

In [3]:
# Confirm that the notebook is operating on the same raw dataset established in Notebook 1
if df.shape != (1433, 63):
    raise ValueError(
        f"Unexpected raw dataset shape: {df.shape}. "
        f"Expected (1433, 63)."
    )

print("Raw-data integrity check passed.")

Raw-data integrity check passed.


## 4. Data-Quality Decisions Carried Forward from Data Understanding

Notebook 01 established several findings that directly affect preparation:

- The survey contains a mixture of numerical, binary, nominal, and text-based responses.
- Missingness is substantial and non-uniform rather than evenly distributed across variables.
- Several questions are conditionally applicable, meaning that missingness can reflect questionnaire structure rather than simple random non-response.
- The two `Why or why not?` variables contain very high-cardinality free-text responses.
- Several diagnosis/condition questions also contain high-cardinality textual responses.
- The work-position question contains multiple role descriptions within individual responses.
- The age variable contains clearly implausible values that require correction.
- Exact duplicate rows were not identified in the raw dataset.

These findings provide the evidence base for the preparation decisions below.

## 5. Correction of Demonstrably Invalid Numerical Values

### Question

Which numerical observations must be corrected before numerical representation?

### Evidence

Notebook 01 identified age values that are inconsistent with the intended population of the survey, including observations below adulthood and values that cannot represent a person's age.

For this project, the defensible working-age range is defined as **18–75 years inclusive**.

### Decision

Age values outside the inclusive range 18–75 are converted to missing values rather than deleting the corresponding respondents.

This preserves the respondent record while preventing invalid numerical values from influencing later distance calculations.

In [4]:
# Replace ages outside the project's accepted 18-75 working-age range with missing values.
prepared_df = clean_age(df)

invalid_age_count = df["What is your age?"].ne(
    prepared_df["What is your age?"]
).sum()

print(f"Age values converted to missing: {invalid_age_count}")

Age values converted to missing: 5


### Interpretation

The invalid age values are treated as data-quality errors rather than legitimate observations.

Converting them to missing values allows the later numerical preprocessing pipeline to handle the affected observations without deleting otherwise usable survey responses.

The raw `df` remains unchanged.

## 6. Treatment of Free-Text and High-Cardinality Text Responses

### Question

Which textual responses should be excluded from the primary structured feature representation?

### Evidence

Notebook 01 identified two `Why or why not?` fields containing more than 1,000 distinct responses each. It also identified diagnosis/condition questions containing many distinct textual responses.

These variables are qualitatively different from structured survey categories. Direct one-hot encoding would treat individual textual responses as separate categories and create a sparse, high-dimensional representation with limited interpretability.

### Decision

Raw open-ended responses are excluded from the primary structured clustering representation, including:

- `Why or why not?` responses;
- free-text diagnosis/condition descriptions;
- the raw comments field, if present.

These responses are not considered inherently uninformative. They are outside the primary structured representation because incorporating them responsibly would require a separate text-analysis methodology.

Structured survey questions concerning diagnosis, treatment, workplace attitudes, and support remain available for the analytical representation.

In [7]:
# Remove raw open-text responses from the structured analytical represenattion.
prepared_df = prepared_df.drop(
    columns=[
        column
        for column in prepared_df.columns
        if (
            column.startswith("Why or why not?")
            or "condition(s)" in column.lower()
            or column.lower() == "comments"
        )
    ],
    errors="ignore",
)

print(f"Prepared variables after text exclusion: {prepared_df.shape[1]}")

Prepared variables after text exclusion: 58


## 7. Feature Engineering for Multi-Response Work Position

### Question

How should the work-position question be represented when one respondent can select multiple roles?

### Evidence

Notebook 01 showed that the work-position field contains multiple role descriptions within individual responses and therefore has high observed cardinality.

Treating each complete combination as a separate categorical level would incorrectly represent role combinations as unrelated nominal categories.

### Decision

The response is decomposed into binary component indicators.

A respondent selecting both `Back-end Developer` and `DevOps/SysAdmin`, for example, receives a positive indicator for both roles.

This preserves the multi-response structure while avoiding artificial combination-based categories.

In [8]:
# Replace the multi-response work-position field with interpretable binary code indicators.
prepared_df = engineer_work_position_features(prepared_df)

### Interpretation

The original work-position field no longer needs to be treated as a single high-cardinality categorical variable.

Its component indicators preserve the information that respondents may perform multiple roles while maintaining an interpretable representation for later analysis.

## 8. Variable Representation

The prepared dataset now contains the information considered for the analytical feature matrix.

The representation rules are:

| Variable type | Representation |
| --- | --- |
| Continuous numerical | Numerical representation |
| Binary numerical | Retain as binary information |
| Nominal categorical | One-hot encoding |
| Multi-response categorical | Component binary indicators |
| Genuine ordinal variables | Ordinal encoding only where explicitly defensible |
| Raw free text | Excluded from primary structured representation |

### Ordinal representation

No arbitrary numerical ranking is imposed on nominal survey responses.

Responses such as `Yes`, `No`, and `I don't know` do not establish a complete numerical ordering and therefore remain categorical.

Where a survey question contains a genuine ordered scale, that ordering can be represented explicitly. If the ordering cannot be established without imposing an assumption, nominal encoding is preferred.

## 9. Analytical Preprocessing Pipeline

### Question

How should the prepared survey representation be converted into a numerical feature matrix?

### Method

A preprocessing pipeline applies transformations according to the representation of each variable:

1. **Age** is treated as a continuous numerical variable. Invalid ages have already been converted to missing values, and remaining missing values are median-imputed.
2. **Binary numerical variables** retain their 0/1 representation. Missing responses are imputed using the most frequent observed response, while a missingness indicator preserves information that the original response was absent.
3. **Nominal categorical variables** receive an explicit `Missing` category before one-hot encoding.
4. **Engineered work-position indicators** remain binary numerical features.

Scaling is intentionally kept separate from this pipeline. Scaling is a modelling representation decision that will be evaluated alongside feature selection and dimensionality reduction rather than hidden inside the data-cleaning process.

In [9]:
# Identify the continuous age variable, binary numerical variables,
# and nominal categorical variables.
age_column = "What is your age?"

numeric_columns = prepared_df.select_dtypes(include="number").columns

binary_numeric_columns = [
    column
    for column in numeric_columns
    if column != age_column
    and prepared_df[column].dropna().isin([0, 1]).all()
]

categorical_columns = prepared_df.select_dtypes(
    exclude="number"
).columns

In [10]:
# Define type-specific preprocessing so that each variable
# representation is handled appropriately.
preprocessor = ColumnTransformer(
    transformers=[
        (
            "age",
            SimpleImputer(strategy="median"),
            [age_column],
        ),
        (
            "binary",
            SimpleImputer(
                strategy="most_frequent",
                add_indicator=True,
            ),
            binary_numeric_columns,
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="constant",
                            fill_value="Missing",
                        ),
                    ),
                    (
                        "encoder",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=False,
                        ),
                    ),
                ]
            ),
            categorical_columns,
        ),
    ],
    remainder="drop",
)

## 10. Construct the Analytical Feature Matrix

The preprocessing pipeline is fitted to the prepared survey representation and transforms the data into a numerical feature matrix.

At this stage, the purpose is **representation**, not model fitting.

In [11]:
# Transform the prepared survey into a numerical feature matrix
# and retain traceable feature names.
X_encoded = pd.DataFrame(
    preprocessor.fit_transform(prepared_df),
    columns=preprocessor.get_feature_names_out(),
    index=prepared_df.index,
)

print(
    "Prepared analytical matrix: "
    f"{X_encoded.shape[0]:,} observations x "
    f"{X_encoded.shape[1]} features."
)

Prepared analytical matrix: 1,433 observations x 495 features.


## 11. Preparation Validation

### Question

Does the prepared feature matrix satisfy the structural requirements for subsequent unsupervised analysis?

### Validation criteria

The analytical matrix must:

- preserve the original number of respondents;
- contain only finite numerical values;
- contain no remaining missing values;
- retain traceable feature names.

These checks validate the output of preparation without making any claim that all generated features are appropriate for final clustering.

In [12]:
# Validate the structural properties required before
# feature selection and dimensionality reduction.
if X_encoded.shape[0] != df.shape[0]:
    raise ValueError(
        "The prepared matrix does not preserve the respondent count."
    )

if not np.isfinite(X_encoded.to_numpy()).all():
    raise ValueError(
        "The prepared matrix contains missing or infinite vales."
    )

print(
    "Prepared feature matrix validation passed: "
    f"{X_encoded.shape[0]:,} respondents and "
    f"{X_encoded.shape[1]} finite numerical features."
)

Prepared feature matrix validation passed: 1,433 respondents and 495 finite numerical features.


In [15]:
# Persist the reproducible analytical feature matrix for subsequent notebooks.
processed_path = Path("../data/processed/mental_health_prepared.csv")
processed_path.parent.mkdir(parents=True, exist_ok=True)

X_encoded.to_csv(processed_path, index=True)

print(
    f"Prepared analytical dataset saved to: {processed_path}"
)

Prepared analytical dataset saved to: ..\data\processed\mental_health_prepared.csv


In [16]:
if not processed_path.exists():
    raise FileNotFoundError(
        f"Processed dataset was not created: {processed_path}"
    )

print(
    f"Saved analytical matrix: "
    f"{X_encoded.shape[0]:,} observations x "
    f"{X_encoded.shape[1]} features."
)

Saved analytical matrix: 1,433 observations x 495 features.


## 12. Representation Traceability

The preprocessing pipeline retains source-variable information in the generated feature names. This provides traceability between the encoded analytical matrix and the original survey questions.

The matrix remains a prepared feature space rather than a final clustering feature set. Subsequent analysis will determine which features should be retained and whether dimensionality reduction is appropriate.

In [14]:
# Confirm that the engineered work-position indicators
# were retained in the encoded representation.
work_position_feature_count = X_encoded.filter(
    regex=r"work_position__"
).shape[1]

print(
    "Work-position indicators retained in analytical matrix: "
    f"{work_position_feature_count}"
)

Work-position indicators retained in analytical matrix: 12


## 13. Preparation Decisions and Consequences

The Data Preparation phase has produced a reproducible numerical representation from the raw survey while preserving a clear separation between raw data and analytical transformations.

### Decision 1 — Raw data remains unchanged

The original survey is retained as the immutable source dataset. Preparation operates on a separate representation.

### Decision 2 — Ages outside 18–75 are converted to missing

Values outside the project's defined working-age range are treated as invalid representations of age. The respondents themselves are retained.

### Decision 3 — Raw free text is excluded from the primary structured feature space

The open-ended `Why or why not?` responses and high-cardinality condition descriptions require a dedicated text-analysis approach and are therefore not directly one-hot encoded.

### Decision 4 — Multi-response work positions are decomposed

The work-position field is converted into component indicators so that multiple roles can be represented without treating every combination as a unique category.

### Decision 5 — Binary variables retain binary information

Binary survey responses are preserved as binary information rather than converted into arbitrary ordinal rankings.

### Decision 6 — Categorical missing responses are represented explicitly

Missing categorical responses are represented by a dedicated `Missing` category rather than silently replaced with the most frequent response.

### Decision 7 — Age is treated as continuous

After invalid values are converted to missing, remaining missing age values are median-imputed.

### Decision 8 — Feature selection remains a separate analytical decision

The resulting matrix is a prepared feature space, not an assertion that every generated feature should ultimately enter clustering.

The next stages will therefore evaluate feature relevance, redundancy, representation, and dimensionality before the final clustering experiments.

## Conclusion

The raw OSMI survey has been converted into a structured numerical representation.

The preparation stage addressed the principal issues identified previously: invalid numerical values, high-cardinality textual responses, multi-response categorical structure, heterogeneous variable types, and non-uniform missingness.

The resulting feature matrix preserves the original respondent count, contains finite numerical values, and maintains traceability to the underlying survey variables.

Importantly, the resulting matrix is **not yet treated as the final clustering representation**. Feature-selection and dimensionality-reduction decisions remain necessary before the clustering experiments.

No clustering model or final cluster solution is selected at this stage.